## 1. Introduction <a id='introduction'></a>

### What are Transcription-Based Learning Circuits?

Transcription-based learning circuits are biological computing systems that use gene expression regulation to implement learning and memory. These circuits consist of:

- **mRNA (M)**: Messenger RNA transcripts
- **Protein (P)**: Translated proteins from mRNA
- **Repressor (R)**: Regulatory proteins that control transcription
- **Learning Parameter (H_tot)**: Total concentration of a transcription factor that can be modified during learning

### Key Features

- **Biological Plausibility**: Based on real gene regulatory mechanisms
- **Trainability**: Can learn to respond to stimuli through parameter adaptation
- **Tunability**: Multiple parameters control circuit dynamics
- **Measurability**: Outputs can be measured through protein fluorescence

## 2. Setup and Installation <a id='setup'></a>

First, let's import the necessary modules and set up our environment.

In [ ]:
# Import TCCDP modules
from tccdp.circuits import TranscriptionCircuit
from tccdp.simulators import ODESimulator, step_stimulus, pulse_stimulus, constant_stimulus
from tccdp.training import (
    Trainer,
    pavlovian_protocol,
    sleep_wake_protocol,
    ProgressBarCallback,
    HistoryCallback
)
from tccdp.core import AutoregulationRule, HebbianRule
from tccdp.training.visualize import plot_learning_curves, plot_state_evolution
from tccdp.analysis import compute_performance_metrics, print_metrics_summary

# Standard libraries
import numpy as np
import matplotlib.pyplot as plt

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✓ All modules imported successfully!")

## 3. Creating a Transcription Circuit <a id='creating-circuit'></a>

Let's create our first transcription circuit with default parameters.

In [ ]:
# Create a circuit with default parameters
circuit = TranscriptionCircuit()

print("Circuit created successfully!")
print(f"\nState variables: {circuit.get_state_names()}")
print(f"Initial state: {circuit.get_initial_state()}")
print(f"\nCircuit: {circuit}")

### Customizing Circuit Parameters

You can customize the circuit by providing specific parameters:

In [ ]:
# Create a custom circuit
custom_params = {
    'alpha_m': 10.0,    # mRNA production rate
    'beta_m': 2.0,      # mRNA degradation rate
    'alpha_p': 5.0,     # Protein production rate
    'beta_p': 1.0,      # Protein degradation rate
    'alpha_r': 3.0,     # Repressor production rate
    'beta_r': 1.5,      # Repressor degradation rate
    'k': 2.0,           # Hill coefficient
    'h_tot': 2.5,       # Initial learning parameter
}

custom_circuit = TranscriptionCircuit(params=custom_params)
print("Custom circuit created!")
print(f"H_tot = {custom_circuit.get_param('h_tot')}")

## 4. Running Simulations <a id='simulations'></a>

Now let's simulate the circuit dynamics with different types of stimuli.

### Example 1: Constant Stimulus

In [ ]:
# Create simulator
simulator = ODESimulator(circuit=circuit, dt=0.1, method='RK45')

# Simulate with constant stimulus
results = simulator.simulate(
    duration=50.0,
    stimulus=constant_stimulus(amplitude=1.0)
)

# Plot results
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Plot state variables
ax1.plot(results['time'], results['states'][:, 0], label='mRNA (M)', linewidth=2)
ax1.plot(results['time'], results['states'][:, 1], label='Protein (P)', linewidth=2)
ax1.plot(results['time'], results['states'][:, 2], label='Repressor (R)', linewidth=2)
ax1.set_xlabel('Time')
ax1.set_ylabel('Concentration')
ax1.set_title('State Variables Evolution (Constant Stimulus)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot output
ax2.plot(results['time'], results['outputs'], label='Output', color='red', linewidth=2)
ax2.set_xlabel('Time')
ax2.set_ylabel('Output')
ax2.set_title('Circuit Output')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final output: {results['outputs'][-1]:.4f}")

### Example 2: Step Stimulus

In [ ]:
# Simulate with step stimulus
results_step = simulator.simulate(
    duration=100.0,
    stimulus=step_stimulus(step_time=20.0, amplitude=2.0)
)

# Plot
fig, ax = plt.subplots(figsize=(12, 5))

# Plot stimulus (for reference)
stimulus_values = [step_stimulus(step_time=20.0, amplitude=2.0)(t) for t in results_step['time']]
ax2 = ax.twinx()
ax2.fill_between(results_step['time'], stimulus_values, alpha=0.2, color='gray', label='Stimulus')
ax2.set_ylabel('Stimulus', color='gray')
ax2.tick_params(axis='y', labelcolor='gray')

# Plot output
ax.plot(results_step['time'], results_step['outputs'], label='Output', color='red', linewidth=2)
ax.set_xlabel('Time')
ax.set_ylabel('Output', color='red')
ax.tick_params(axis='y', labelcolor='red')
ax.set_title('Step Response')
ax.grid(True, alpha=0.3)

fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9))
plt.tight_layout()
plt.show()

### Example 3: Pulse Stimulus

In [ ]:
# Simulate with pulse stimulus
results_pulse = simulator.simulate(
    duration=100.0,
    stimulus=pulse_stimulus(start_time=20.0, end_time=40.0, amplitude=3.0)
)

# Plot
plt.figure(figsize=(12, 5))
plt.plot(results_pulse['time'], results_pulse['outputs'], linewidth=2, color='purple')
plt.xlabel('Time')
plt.ylabel('Output')
plt.title('Pulse Response (Pulse from t=20 to t=40)')
plt.grid(True, alpha=0.3)
plt.axvline(20, color='gray', linestyle='--', alpha=0.5, label='Pulse start')
plt.axvline(40, color='gray', linestyle='--', alpha=0.5, label='Pulse end')
plt.legend()
plt.show()

## 5. Training with Pavlovian Conditioning <a id='training'></a>

Now let's train the circuit using Pavlovian conditioning protocol. The circuit will learn to respond to a conditioned stimulus (CS).

In [ ]:
# Create a fresh circuit for training
training_circuit = TranscriptionCircuit()
training_simulator = ODESimulator(circuit=training_circuit, dt=0.1)

# Create Pavlovian protocol
protocol = pavlovian_protocol(
    cs_duration=10.0,   # Conditioned stimulus duration
    us_duration=5.0,    # Unconditioned stimulus duration
    iti_duration=15.0   # Inter-trial interval
)

print(f"Protocol cycle duration: {protocol.cycle_duration()} time units")
print(f"Number of phases: {len(protocol.phases)}")

In [ ]:
# Create trainer with Autoregulation learning rule
trainer = Trainer(
    circuit=training_circuit,
    simulator=training_simulator,
    learning_rule=AutoregulationRule(learning_rate=0.01),
    target_function=lambda t: 1.0  # Target output is 1.0
)

# Train the circuit
print("\nStarting training...\n")
history = trainer.train(
    protocol=protocol,
    n_epochs=100,
    verbose=True,
    callbacks=[ProgressBarCallback(), HistoryCallback()]
)

print(f"\n✓ Training completed!")
print(f"Final loss: {history['losses'][-1]:.6f}")
print(f"Initial loss: {history['losses'][0]:.6f}")
print(f"Improvement: {(1 - history['losses'][-1]/history['losses'][0])*100:.1f}%")

## 6. Visualizing Results <a id='visualization'></a>

Let's visualize the training progress and results.

In [ ]:
# Plot learning curves
fig = plot_learning_curves(history, show=False)
plt.show()

In [ ]:
# Plot learning parameter evolution
plt.figure(figsize=(12, 5))
plt.plot(history['epochs'], history['learning_params'], linewidth=2, color='green')
plt.xlabel('Epoch')
plt.ylabel('H_tot (Learning Parameter)')
plt.title('Learning Parameter Evolution')
plt.grid(True, alpha=0.3)
plt.axhline(y=history['learning_params'][0], color='r', linestyle='--', alpha=0.5, label='Initial')
plt.axhline(y=history['learning_params'][-1], color='b', linestyle='--', alpha=0.5, label='Final')
plt.legend()
plt.show()

print(f"H_tot changed from {history['learning_params'][0]:.4f} to {history['learning_params'][-1]:.4f}")

In [ ]:
# Compare initial vs final response
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot first 10 epochs
for i in range(min(10, len(history['outputs']))):
    alpha = 0.3 + 0.7 * (i / 10)
    ax1.plot(history['outputs'][i], alpha=alpha, color='blue')
ax1.set_title('First 10 Epochs')
ax1.set_xlabel('Time Step')
ax1.set_ylabel('Output')
ax1.grid(True, alpha=0.3)

# Plot last 10 epochs
start_idx = max(0, len(history['outputs']) - 10)
for i in range(start_idx, len(history['outputs'])):
    alpha = 0.3 + 0.7 * ((i - start_idx) / 10)
    ax2.plot(history['outputs'][i], alpha=alpha, color='red')
ax2.set_title('Last 10 Epochs')
ax2.set_xlabel('Time Step')
ax2.set_ylabel('Output')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Performance Metrics <a id='metrics'></a>

Let's compute comprehensive performance metrics for our trained circuit.

In [ ]:
# Compute performance metrics
metrics = compute_performance_metrics(history, target_value=1.0)

# Print summary
summary = print_metrics_summary(metrics)
print(summary)

In [ ]:
# Visualize key metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# MSE over epochs
mse_values = [(o - 1.0)**2 for o in [np.mean(outputs) for outputs in history['outputs']]]
axes[0, 0].plot(history['epochs'], mse_values, linewidth=2, color='red')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('MSE')
axes[0, 0].set_title('Mean Squared Error Evolution')
axes[0, 0].grid(True, alpha=0.3)

# Loss over epochs (log scale)
axes[0, 1].semilogy(history['epochs'], history['losses'], linewidth=2, color='blue')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss (log scale)')
axes[0, 1].set_title('Loss Convergence')
axes[0, 1].grid(True, alpha=0.3)

# Convergence rate indicator
window = 10
if len(history['losses']) > window:
    convergence_indicator = []
    for i in range(window, len(history['losses'])):
        std = np.std(history['losses'][i-window:i])
        mean = np.mean(history['losses'][i-window:i])
        convergence_indicator.append(std / (mean + 1e-10))
    axes[1, 0].plot(history['epochs'][window:], convergence_indicator, linewidth=2, color='green')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Convergence Indicator')
    axes[1, 0].set_title('Loss Variation (Lower = More Converged)')
    axes[1, 0].axhline(y=0.01, color='r', linestyle='--', alpha=0.5, label='Threshold')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

# Metrics summary bar chart
metric_names = ['Final\nLoss', 'MSE', 'RMSE', 'MAE']
metric_values = [metrics.final_loss, metrics.mse, metrics.rmse, metrics.mae]
axes[1, 1].bar(metric_names, metric_values, color=['red', 'orange', 'yellow', 'green'], alpha=0.7)
axes[1, 1].set_ylabel('Value')
axes[1, 1].set_title('Error Metrics Summary')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 8. Advanced Examples <a id='advanced'></a>

### Example 1: Comparing Different Learning Rules

In [ ]:
# Compare Autoregulation vs Hebbian learning
learning_rules = {
    'Autoregulation': AutoregulationRule(learning_rate=0.01),
    'Hebbian': HebbianRule(learning_rate=0.01)
}

results_comparison = {}

for name, rule in learning_rules.items():
    print(f"\nTraining with {name} rule...")
    
    # Create fresh circuit
    circuit = TranscriptionCircuit()
    simulator = ODESimulator(circuit=circuit, dt=0.1)
    
    # Train
    trainer = Trainer(circuit, simulator, rule, target_function=lambda t: 1.0)
    history = trainer.train(protocol, n_epochs=50, verbose=False)
    
    results_comparison[name] = history
    print(f"  Final loss: {history['losses'][-1]:.6f}")

# Plot comparison
plt.figure(figsize=(12, 5))
for name, history in results_comparison.items():
    plt.plot(history['epochs'], history['losses'], linewidth=2, label=name, marker='o', markersize=3)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Learning Rule Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

### Example 2: Parameter Sensitivity Analysis

In [ ]:
# Test different learning rates
learning_rates = [0.001, 0.005, 0.01, 0.05, 0.1]
lr_results = {}

for lr in learning_rates:
    circuit = TranscriptionCircuit()
    simulator = ODESimulator(circuit=circuit, dt=0.1)
    trainer = Trainer(circuit, simulator, AutoregulationRule(learning_rate=lr), lambda t: 1.0)
    history = trainer.train(protocol, n_epochs=50, verbose=False)
    lr_results[lr] = history
    print(f"LR={lr:.3f}: Final loss = {history['losses'][-1]:.6f}")

# Plot
plt.figure(figsize=(12, 5))
for lr, history in lr_results.items():
    plt.plot(history['epochs'], history['losses'], linewidth=2, label=f'LR={lr:.3f}')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Learning Rate Sensitivity')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

## Summary

In this tutorial, you learned:

1. ✅ **Created** transcription-based learning circuits
2. ✅ **Simulated** circuit dynamics with various stimuli
3. ✅ **Trained** circuits using Pavlovian conditioning
4. ✅ **Visualized** learning progress and state evolution
5. ✅ **Analyzed** performance using comprehensive metrics
6. ✅ **Compared** different learning rules and parameters

### Next Steps

- Explore other circuit types (protein modification, metabolic)
- Try different training protocols (sleep-wake cycles)
- Experiment with stochastic simulations (Gillespie algorithm)
- Design custom learning rules
- Integrate multiple circuits

### Resources

- [TCCDP Documentation](https://github.com/morningpython/trainable_cell_circuits_design)
- [API Reference](https://tccdp.readthedocs.io)
- [More Examples](../examples/)

In [ ]:
# Save your results
# import json
# with open('training_history.json', 'w') as f:
#     json.dump(history, f, indent=2, default=lambda x: x.tolist() if isinstance(x, np.ndarray) else x)
# print("Results saved to training_history.json")